# 소개: PhysicsNeMo의 개념 · 한국어판

[시작 안내](Start_Here.ipynb) · 이전: [환경 설정](00_Setup.ipynb) · 다음: [실습 1: PINN 기초](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)

물리 정보 신경망(PINN)은 해를 예측하고, 방정식과 초기 조건 또는 경계 조건을 만족하도록 학습합니다. 여기에서 이러한 구성 요소를 익힌 뒤 [실습 1](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)에서 모델을 학습합니다.

이 노트북은 읽기용입니다. 코드를 실행하는 연습은 실습 1부터 시작합니다.

## PhysicsNeMo 소개

PhysicsNeMo는 물리 기반 학습을 위한 PyTorch 모델과 도구를 제공합니다. 이 과정은 버전 2.2.2를 사용하며, PINN에는 완전 연결 신경망을, 신경 연산자에는 FNO와 AFNO를, 물리 손실에는 기호식으로 표현한 방정식을 사용합니다.

[공식 개요](https://docs.nvidia.com/physicsnemo/latest/index.html) · [버전 2.2.2 소스](https://github.com/NVIDIA/physicsnemo/tree/v2.2.2) · [버전 2.0 마이그레이션 안내](https://github.com/NVIDIA/physicsnemo/blob/v2.2.2/v2.0-MIGRATION-GUIDE.md)

### 이 과정에서 사용하는 구성 요소

| 구성 요소 | 역할 | 이 과정의 사용 예 |
|---|---|---|
| `physicsnemo.models` | PyTorch 모델 구조 | `FullyConnected`, 이후 FNO와 AFNO |
| SymPy | 장과 미분을 기호식으로 표현 | `Function("u")(x, y, t)`, `u.diff(x, 2)` |
| `physicsnemo.sym` | 방정식 정의를 저장하고 잔차 텐서 계산 | `PDE`, `PhysicsInformer` |
| `ETC/runtime`의 강의 보조 함수 | 수업 설정과 텐서를 라이브러리에 연결 | PINN 도전 과제의 `create_model`, `create_informer`, `residuals` |
| 데이터 및 좌표 샘플링 | 학습 입력 준비 | `torch.rand`, 초기 조건 데이터 |
| 명시적인 학습 루프 | 손실 계산 및 매개변수 최적화 | `torch.optim.Adam` 또는 `torch.optim.LBFGS`, `backward`, `step` |
| 검증 및 추론 | 방정식, 경계 조건, 예측 오차 확인 | 고정된 평가 좌표; 실습 4 예측을 ERA5 및 지속성 예측과 비교 |

Sym은 Symbolic(기호식)의 약자입니다. SymPy는 방정식을 표현하고, `PhysicsInformer`는 예측한 장의 PyTorch 텐서로 그 방정식을 계산합니다. 입문용 MLP 연습은 CPU에서도 실행됩니다. 호환되는 CUDA 지원 PyTorch 환경에서는 `--device cuda`를 선택하면 GPU를 사용할 수 있습니다.

### 강의 보조 함수 살펴보기

`create_model`, `create_informer`, `residuals`는 [ETC/runtime/pinn.py](ETC/runtime/pinn.py)에 정의된 강의용 유틸리티이며 PhysicsNeMo의 공개 API가 아닙니다. `create_model`은 `physicsnemo.models.mlp.FullyConnected`를 생성하고, `create_informer`는 강의용 어댑터를 통해 `physicsnemo.sym.eq.phy_informer.PhysicsInformer`를 설정합니다. `residuals`는 신경망을 계산하고 이름이 지정된 텐서를 준비한 뒤 `informer.forward`를 호출합니다. 모델은 장의 값을 예측하고, informer는 방정식 잔차를 반환하며, 학습 루프는 잔차를 하나의 손실값으로 집계합니다.

실습 1–3은 [ETC/runtime/labs.py](ETC/runtime/labs.py)의 `mlp`, `informer`와 같은 관련 보조 함수를 사용하지만, 인자와 모델 코드는 수업별로 다릅니다. 실습 4는 기상 추론을 위해 사전 학습된 AFNO 모델을 불러오며 PINN 학습 루프를 사용하지 않습니다. 모든 수업이 도전 과제 보조 함수와 같은 인자를 사용한다고 가정하지 말고, 각 파일의 import를 따라가세요.

`student_equations`부터 `total.backward()`까지의 전체 동작 과정은 [강의 보조 함수에서 PhysicsNeMo까지](ETC/course_materials/PHYSICSNEMO_WORKFLOW.md)를 읽어보세요. 파동 1단계를 따라 설명하며, 이후 도전 과제 4에서 사용하는 연산자 실행 흐름도 다룹니다.

### PINN 수업의 모델 학습 과정

1. 좌표, 예측할 물리량, 영역, 단위를 지정합니다.
2. PhysicsNeMo의 `PDE` 하위 클래스에 SymPy 식을 작성하고 `FullyConnected` 신경망을 생성합니다.
3. 좌표를 샘플링하고 방정식, 조건, 관측값이 있다면 관측값에 대한 손실도 계산합니다.
4. 고정된 격자에서 평가하고 학습과 겹치는 좌표나 매개변숫값을 확인합니다.
5. Python 기본값이나 YAML 설정에서 해당 수업의 학습 횟수, 배치 크기, 학습률, 신경망 크기를 확인합니다.
6. 해당 수업의 옵티마이저로 학습합니다. 실습 1은 L-BFGS, 실습 2는 Adam, 실습 3은 Adam 이후 L-BFGS를 사용합니다. 예측, 잔차, 해석해가 있으면 해석해도 비교합니다. 실습 4에는 옵티마이저가 없으며, 사전 학습된 기상 모델을 실행합니다.

실습 1–3은 명시적인 PyTorch 학습 루프와 수업별 설정을 사용합니다. 이는 [PhysicsNeMo 공식 물리 기반 학습 흐름](https://docs.nvidia.com/physicsnemo/latest/user-guide/physics_addition.html)과 같습니다. PhysicsNeMo가 모델과 물리 도구를 제공하고, 직접 작성한 루프가 샘플링, 손실, 최적화를 제어합니다. 연결된 Python 파일에서 이러한 설정을 살펴볼 수 있습니다.

## PINN, 신경 연산자, PINO

데이터 기반 모델은 입력과 목표값의 쌍으로 학습합니다. 도전 과제 4에서 FNO와 AFNO는 강제항의 장을 입력받아 해의 장을 예측합니다. PINN은 좌표를 입력받고, 방정식 및 조건 손실과 경우에 따라 관측값을 이용해 해를 학습합니다. PINO는 신경 연산자 학습에 물리 잔차를 추가합니다.

[실습 4: FourCastNet을 활용한 AI 기상 예측](01_labs/04_weather_forecasting/Lab_4_Weather_Forecasting.ipynb)에서는 이미 학습된 AFNO가 전 지구 기상장 26개를 6시간 뒤의 동일한 기상장으로 매핑합니다. 각 예측을 다음 입력으로 다시 사용하여 48시간 예측을 생성합니다. 예측에는 최초 ERA5 상태만 입력하며, 이후 시각의 ERA5 재분석 자료는 검증용으로만 사용합니다. 초기 기상 상태를 그대로 유지하는 지속성 기준 예측과 비교하세요. ERA5는 관측과 수치 모델을 결합한 자료이며, 직접 관측만으로 이루어진 것은 아닙니다.

이 과정은 사전 학습 모델의 추론이며, 처음부터 모델을 학습하거나 PDE 잔차를 최소화하는 과정이 아닙니다. 과거 기상 사례 하나로 실행 흐름을 살펴보는 것이며, 전반적인 예측 신뢰도를 입증하지는 않습니다. 기존 Navier–Stokes 실습은 [참고 자료](ETC/reference_labs/04_navier_stokes/Lab_4_Navier_Stokes.ipynb)로 보존되어 있습니다.

`FullyConnected`, FNO, AFNO는 모델 구조를 뜻합니다. PINN과 PINO는 물리 정보를 학습에 반영하는 방법을 뜻합니다. 같은 모델도 서로 다른 손실로 학습할 수 있으므로 예측 오차와 만족해야 할 방정식을 모두 확인하세요.

도전 과제 4는 공개 클래스인 `physicsnemo.models.fno.FNO`와 `physicsnemo.models.afno.AFNO`를 사용합니다. PINO 단계에서는 주기 격자에서 `grad_method="spectral"`로 설정한 `PhysicsInformer`를 추가합니다. 데이터 손실은 정규화된 장을 비교하며, PDE 잔차는 예측과 강제항을 물리적 스케일로 복원한 뒤 계산합니다.

### 첫 번째 문제에 사용할 신경망

```python
from physicsnemo.models.mlp import FullyConnected
model = FullyConnected(in_features=1, out_features=1, layer_size=64,
                       num_layers=3, activation_fn="tanh")
```

다음 실습에서는 이 모델을 PDE와 학습 루프에 연결합니다. `tanh`와 같이 매끄러운 활성화 함수는 2차 미분이 필요한 PINN에 적합합니다.

### 두 종류의 미분

신경망은 좌표 $x$를 입력받아 $u_\theta(x)$를 예측합니다. `PhysicsInformer`는 공간 좌표에 대해 미분하여 잔차 $u_{\theta,xx}-1$을 만듭니다. 손실은 이 잔차와 양 끝 경계에서의 오차를 결합합니다.

이어서 `loss.backward()`가 신경망 매개변수 $\theta$에 대해 손실을 미분하고, `optimizer.step()`이 그 매개변수를 갱신합니다. 좌표는 입력이며 옵티마이저가 변경하는 양이 아닙니다.

시간 의존 PINN 도전 과제에서는 강의의 `residuals` 보조 함수가 `torch.autograd.grad`로 `u__t`와 `u__t__t`를 계산하여 텐서를 명시적으로 전달하고, `PhysicsInformer`가 공간 미분을 처리합니다. 잔차는 해의 오차가 아니라 방정식의 불일치 정도를 측정합니다. 예를 들어 값이 모두 0인 장은 동차 파동 PDE를 만족하지만, 파동 1단계의 0이 아닌 초기 조건은 만족하지 않습니다.

[pinn_basics.py](01_labs/01_pinn/source_code/pinn_basics.py)의 `BasicPINN`, `loss_terms`, `optimize_lab`에서 이 연산들을 따라갈 수 있습니다. L-BFGS 갱신은 `optimizer.step(closure)`를 사용하며, closure는 고정된 학습점에서 손실과 기울기를 다시 계산합니다. 양 끝 값이 0인 $u''=1$의 곡선은 0보다 위에 있을까요, 아래에 있을까요? 해석해는 비교에 사용하며 학습 목표값으로 사용하지 않습니다.

### 다음: 실습 1

[PINN 기초](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)를 열어 첫 번째 예제를 실행하세요.

[시작 안내](Start_Here.ipynb) · 다음: [실습 1: PINN 기초](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)

--- 

추가 자료: [Open Hackathons](https://www.openhackathons.org/s/technical-resources), [OpenACC 및 Hackathons Slack 채널](https://www.openacc.org/community#slack).

---

# 라이선스

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.